# 11 Full Session Walkthrough

Build a complete session with transport links, sequencers, sampler routing, mixer inserts, and a timeline lane ready for browser recording.


In [ ]:
import math

import ipywidgets as widgets
from IPython.display import display

from nbplay import (
    EffectPlugin,
    KeyboardWidget,
    PadWidget,
    SamplerWidget,
    SequencerWidget,
    Session,
    SettingsWidget,
    SynthWidget,
)

settings = SettingsWidget(sample_rate=44100, channels=2, buffer_size=512)

lead = SynthWidget(oscillator_type="saw", frequency=660.0, amplitude=0.55)
bass = SynthWidget(oscillator_type="square", frequency=110.0, amplitude=0.65)

drum_sampler = SamplerWidget(
    attack=0.005,
    decay=0.08,
    sustain=0.15,
    release=0.12,
    max_voices=8,
    pad_count=8,
)
sample_rate = 44100
frame_count = int(sample_rate * 0.18)
sample_data = [
    math.sin(2 * math.pi * 880 * index / sample_rate) * (1 - index / frame_count)
    for index in range(frame_count)
]
drum_sampler.load_sample(sample_data, sample_rate=sample_rate, root_note=60, name="Notebook Blip")

lead_seq = SequencerWidget(length=8, bpm=118.0)
for step_index, note in enumerate([72, 76, 79, 83, 79, 76, 74, 71]):
    lead_seq.set_step(step_index, note=note, velocity=102, active=True)

bass_seq = SequencerWidget(length=8, bpm=118.0)
for step_index, velocity in enumerate([118, 64, 88, 64, 110, 64, 92, 64]):
    bass_seq.set_step(step_index, note=48, velocity=velocity, active=True)

drum_seq = SequencerWidget(length=8, bpm=118.0)
for step_index in [0, 3, 4, 7]:
    drum_seq.set_step(step_index, note=60, velocity=112, active=True)

session = Session(bpm=118.0, time_signature=(4, 4))
lead_track = session.add_track("Lead", lead_seq, lead)
bass_track = session.add_track("Bass", bass_seq, bass)
drum_track = session.add_track("Drums", drum_seq, drum_sampler)

session.mixer.set_channel_gain(lead_track.mixer_channel, 0.78)
session.mixer.set_channel_pan(lead_track.mixer_channel, 0.25)
session.mixer.add_channel_effect(
    lead_track.mixer_channel,
    EffectPlugin("filter", filter_type="lowpass", frequency=4200, q=0.8),
)

session.mixer.set_channel_gain(bass_track.mixer_channel, 0.92)
session.mixer.set_channel_pan(bass_track.mixer_channel, -0.18)
session.mixer.add_channel_effect(
    bass_track.mixer_channel,
    EffectPlugin("compressor", threshold=-22, ratio=5),
)

session.mixer.set_channel_gain(drum_track.mixer_channel, 0.58)
session.mixer.add_channel_effect(
    drum_track.mixer_channel,
    EffectPlugin("reverb", seconds=0.9, decay=2.0, wet=0.14),
)
session.mixer.master_gain = 0.86
session.mixer.add_master_effect(EffectPlugin("limiter", threshold=-1))

session.timeline.length = 16
session.timeline.arm_track(drum_track.mixer_channel, True, exclusive=True)
session.timeline.add_clip("Arrangement marker", track_index=drum_track.mixer_channel, start=0, duration=8, source="placeholder")

keyboard = KeyboardWidget(upper_octave=4, lower_octave=3, velocity=96)
keyboard.connect_sequencer(lead_seq)
keyboard.connect_sampler(drum_sampler)

pads = PadWidget(rows=2, cols=4, velocity=112)
pads.connect_sampler(drum_sampler)

tabs = widgets.Tab([
    widgets.VBox([settings, session.transport, session.timeline]),
    widgets.VBox([lead, bass, drum_sampler]),
    widgets.VBox([lead_seq, bass_seq, drum_seq]),
    widgets.VBox([keyboard, pads]),
    session.mixer,
])
for index, title in enumerate(["Transport", "Sources", "Sequencers", "Inputs", "Mixer"]):
    tabs.set_title(index, title)

display(tabs)
print(session)


Arm a timeline track and press record to capture microphone audio into the browser timeline. Recorded clip blobs are browser-local object URLs for now, so rerunning or reloading the notebook clears recorded audio.
